## 概要

実習3_5です。与えられたデータを基に学習を深めてみましょう。

## 業務シナリオの紹介

あなたはある医療機関に勤務しており、乳癌（にゅうがん、英: Breast cancer）の検出を改善したいと考えています。

機械学習 (ML) を利用してこの問題を解決するのがあなたの課題です。このデータセットを使用して ML モデルのトレーニングを行い、患者に異常があるかどうかを予測します。

## このデータセットについて

breast_cancer.csv　は、Pythonのscikit-learn付属データセットに用意されており、ダウンロードしたデータです。

## 属性情報

PowerPointをご覧ください。
このデータセットの詳細については、 (https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) を参照してください。

## データのインポート

以下の3つのセルは、前回の3-4のコードをまとめたものです。
実行するとデータがインポートされ、使用できる状態になります。

**注:** 次のセルは、以前のラボでの重要なステップです。消さないように注意
**fit** を実行するとモデルがトレーニングされます。
**注:** この処理は最長 5 分かかることがあります。

In [1]:
# S3のバケット名を指定するための変数を作成（ご自身のパケットに変更）
bucket=''

In [2]:
# 警告メッセージを無視するためにwarningsモジュールを使用
import warnings, requests, zipfile, io

# 警告メッセージを無視するフィルターを設定
warnings.simplefilter('ignore')

# データ処理のためにpandasモジュールをインポート
import pandas as pd

# 科学技術計算のためにscipyモジュールからarffファイルの入出力関数をインポート
from scipy.io import arff

# ローカルのオペレーティングシステムにアクセスするためにosモジュールをインポート
import os

# AWSのSageMakerサービスを使用するためにboto3とsagemakerモジュールをインポート
import boto3
import sagemaker

# SageMakerにおけるデフォルトのコンテナイメージを取得するためにretrieveメソッドを使用
from sagemaker.image_uris import retrieve

# データをトレーニングセットとテストセットに分割するためにtrain_test_splitメソッドを使用
from sklearn.model_selection import train_test_split

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [3]:
# pandasライブラリを使って、AWS S3上のCSVファイルを読み込む
# 事前に必要なデータをS3にアップロードをして、URIをメモしておいてください。
df = pd.read_csv('')

# DataFrameの列名をリストに変換
cols = df.columns.tolist()

# 先頭の列を末尾に移動するためにスライスを使用
cols = cols[-1:] + cols[:-1]

# 列の順序を変更するためにDataFrameを再構築
df = df[cols]

# データセットをトレーニングセットとテスト＆検証セットに分割するためにtrain_test_split関数を使用
# test_sizeでテスト＆検証セットの割合を指定し、random_stateで乱数のシードを固定
# stratifyでクラスの分布を保持して分割
train, test_and_validate = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

# train_test_split関数を使用してデータを分割
# test_and_validateデータをtestとvalidateに分割し、test_sizeで分割比率を指定
# random_stateで再現性を確保し、stratifyで層別サンプリングを行う（'class'列を考慮）
test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['target'])

# S3バケットとプレフィックスの設定
prefix='lab3'

# 訓練データ、テストデータ、検証データのファイル名設定
train_file='vertebral_train.csv'
test_file='vertebral_test.csv'
validate_file='vertebral_validate.csv'

# S3リソースを取得
s3_resource = boto3.Session().resource('s3')

# S3にCSVファイルをアップロードするための関数を定義
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())
    
# train_fileというCSVファイルをS3にアップロードするためにupload_s3_csv関数を使用
upload_s3_csv(train_file, 'train', train)

# test_fileというCSVファイルをS3にアップロードするためにupload_s3_csv関数を使用
upload_s3_csv(test_file, 'test', test)

# validate_fileというCSVファイルをS3にアップロードするためにupload_s3_csv関数を使用
upload_s3_csv(validate_file, 'validate', validate)

# SageMakerにおいてXGBoostアルゴリズムを利用するためのコンテナイメージを取得
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')

# XGBoostのハイパーパラメータを設定するための辞書を作成
hyperparams={"num_round":"42",
             "eval_metric": "auc",
             "objective": "binary:logistic"}

# 出力先のS3ロケーションを設定
s3_output_location="s3://{}/{}/output/".format(bucket,prefix)

# SageMakerのXGBoostアルゴリズムを使用するためにXGBoost Estimatorを作成
# container: 使用するコンテナの指定
# sagemaker.get_execution_role(): SageMakerの実行ロールを取得
# instance_count: モデルトレーニングに使用するインスタンスの数を指定
# instance_type: モデルトレーニングに使用するインスタンスのタイプを指定
# output_path: トレーニングアーティファクト（モデルアーティファクト）のS3出力先を指定
# hyperparameters: XGBoostのハイパーパラメータを指定
# sagemaker_session: SageMakerセッションを指定
xgb_model=sagemaker.estimator.Estimator(container,
                                       sagemaker.get_execution_role(),
                                       instance_count=1,
                                       instance_type='ml.m4.xlarge',
                                       output_path=s3_output_location,
                                        hyperparameters=hyperparams,
                                        sagemaker_session=sagemaker.Session())

# SageMakerのトレーニングジョブに使用するトレーニングデータのS3パスとコンテントタイプを指定
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

# SageMakerのトレーニングジョブに使用する検証データのS3パスとコンテントタイプを指定
validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix,validate_file),
    content_type='text/csv')

# トレーニングジョブのデータチャネルにトレーニングデータと検証データを指定
data_channels = {'train': train_channel, 'validation': validate_channel}

# XGBoostモデルを学習させるためにfitメソッドを使用
# inputsにはデータチャネルを指定し、logsをFalseに設定してログを表示しないようにする
xgb_model.fit(inputs=data_channels, logs=False)

# ホスティングの準備が完了したことを表示
print('ready for hosting!')

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2024-01-07-01-04-32-327



2024-01-07 01:04:32 Starting - Starting the training job.....
2024-01-07 01:05:07 Starting - Preparing the instances for training..............
2024-01-07 01:06:23 Downloading - Downloading input data.....
2024-01-07 01:06:53 Downloading - Downloading the training image........
2024-01-07 01:07:39 Training - Training image download completed. Training in progress.....
2024-01-07 01:07:59 Uploading - Uploading generated training model..
2024-01-07 01:08:16 Completed - Training job completed
ready for hosting!


# ステップ 1: モデルのホスティング

モデルのトレーニングが完了したので、Amazon SageMaker ホスティングサービスを使用して、モデルをホストします。

最初のステップはモデルのデプロイです。モデルオブジェクトの *xgb_model* があるので、**deploy** メソッドを使用できます。このラボでは、単一の ml.m4.xlarge インスタンスを使用します。
**注:** この処理は最長 5 分かかることがあります。

In [4]:
# XGBoostモデルをデプロイしてエンドポイントを作成
xgb_predictor = 



# ホスティングが完了したことを表示
print('Hosting completed!')

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2024-01-07-01-09-07-311
INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2024-01-07-01-09-07-311
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2024-01-07-01-09-07-311


-------!Hosting completed!


# ステップ 2: 予測の実行

モデルがデプロイされたので、いくつかの予測を実行します。

まずは、テストデータを見直して、もう一度内容をつかんでおきます。

In [5]:
# DataFrameの行数と列数を確認するためにshape属性を使用


(57, 31)

31の属性を持つインスタンスが57個あります。最初の 5 つのインスタンスは次のとおりです。

In [6]:
# テストデータの最初の5行を表示するためにheadメソッドを使用


,target,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
415,1,11.89,21.17,76.39,433.8,0.09773,0.08120,0.02555,0.02179,0.2019,...,13.050,27.21,85.09,522.9,0.14260,0.21870,0.11640,0.08263,0.3075,0.07351
235,1,14.03,21.25,89.79,603.4,0.09070,0.06945,0.01462,0.01896,0.1517,...,15.330,30.28,98.27,715.5,0.12870,0.15130,0.06231,0.07963,0.2226,0.07617
568,1,7.76,24.54,47.92,181.0,0.05263,0.04362,0.00000,0.00000,0.1587,...,9.456,30.37,59.16,268.6,0.08996,0.06444,0.00000,0.00000,0.2871,0.07039
488,1,11.68,16.17,75.49,420.5,0.11280,0.09263,0.04279,0.03132,0.1853,...,13.320,21.59,86.57,549.8,0.15260,0.14770,0.14900,0.09815,0.2804,0.08024
14,0,13.73,22.61,93.60,578.3,0.11310,0.22930,0.21280,0.08025,0.2069,...,15.030,32.01,108.80,697.7,0.16510,0.77250,0.69430,0.22080,0.3596,0.14310


ターゲット値 (target) を含める必要はありません。この予測子は、カンマ区切り値 (CSV) 形式のデータを取り入れます。したがって、*target 列を含めずに* 先頭行を取得するには、次のコードを使用します。

`test.iloc[:1,1:]` 

**iloc** 関数は、[*rows*,*cols*] のパラメータを取ります

先頭行だけを取得するには、`0:1` を使用します。2 行目を取得したい場合は、`1:2` を使用できます。

先頭列 (*col 0*) を *除く* すべての列を取得するには、`1:` を使用します。



In [7]:
# テストデータから1行目（0から始まる行）を取得し、最初の列以降のデータを表示
row = 

# 表示されたデータの最初の行を確認


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
415,11.89,21.17,76.39,433.8,0.09773,0.0812,0.02555,0.02179,0.2019,0.0629,...,13.05,27.21,85.09,522.9,0.1426,0.2187,0.1164,0.08263,0.3075,0.07351


これをカンマ区切り値 (CSV) ファイルに変換し、文字列バッファに格納できます。

In [8]:
# CSV形式の文字列を一時的に格納するStringIOオブジェクトを作成
batch_X_csv_buffer = 

# DataFrameの1行をCSV形式でStringIOに書き込む（ヘッダーなし、インデックスなし）


# StringIOからCSV形式の文字列を取得


# 取得したCSV形式の文字列を表示


11.89,21.17,76.39,433.8,0.09773,0.0812,0.02555,0.02179,0.2019,0.0629,0.2747,1.203,1.93,19.53,0.009895,0.03053,0.0163,0.009276,0.02258,0.002272,13.05,27.21,85.09,522.9,0.1426,0.2187,0.1164,0.08263,0.3075,0.07351



これで、データを使用して予測を実行できるようになりました。

In [9]:
# XGBoostモデルを使用してテストデータの行を予測するためにpredictメソッドを使用


b'0.9978572726249695'

表示される結果は *0* や *1* ではなく、代わりに *確率スコア* が表示されます。確率スコアにいくつかの条件付きロジックを適用することで、答えを 0 または 1 として示すべきかどうか判断できます。バッチ予測を行うときは、この処理を使用します。

ここでは、結果をテストデータと比較します。

In [10]:
# テストデータの最初の5行を表示するためにheadメソッドを使用


,target,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
415,1,11.89,21.17,76.39,433.8,0.09773,0.08120,0.02555,0.02179,0.2019,...,13.050,27.21,85.09,522.9,0.14260,0.21870,0.11640,0.08263,0.3075,0.07351
235,1,14.03,21.25,89.79,603.4,0.09070,0.06945,0.01462,0.01896,0.1517,...,15.330,30.28,98.27,715.5,0.12870,0.15130,0.06231,0.07963,0.2226,0.07617
568,1,7.76,24.54,47.92,181.0,0.05263,0.04362,0.00000,0.00000,0.1587,...,9.456,30.37,59.16,268.6,0.08996,0.06444,0.00000,0.00000,0.2871,0.07039
488,1,11.68,16.17,75.49,420.5,0.11280,0.09263,0.04279,0.03132,0.1853,...,13.320,21.59,86.57,549.8,0.15260,0.14770,0.14900,0.09815,0.2804,0.08024
14,0,13.73,22.61,93.60,578.3,0.11310,0.22930,0.21280,0.08025,0.2069,...,15.030,32.01,108.80,697.7,0.16510,0.77250,0.69430,0.22080,0.3596,0.14310


**質問:** 予測は正確ですか?

**チャレンジ課題:** 以前のコードを更新して、データセットの 2 行目を送信します。それらの予測は正確ですか? 他の行でも、このタスクを試してみましょう。

これらの行を 1 つずつ送信するのは面倒です。この値を一括で送信する関数を記述するという方法もありますが、SageMaker はすでにバッチ機能を備えています。その機能を次に調べます。ただし、実行する前にモデルを終了してください。

# ステップ 3: デプロイしたモデルの終了

エンドポイントを削除するには、予測子で **delete_endpoint** 関数を使用します。

In [11]:
# XGBoost Predictorのエンドポイントとエンドポイント設定を削除するためにdelete_endpointメソッドを使用


INFO:sagemaker:Deleting endpoint configuration with name: sagemaker-xgboost-2024-01-07-01-09-07-311
INFO:sagemaker:Deleting endpoint with name: sagemaker-xgboost-2024-01-07-01-09-07-311


# ステップ 4: バッチ変換の実行

トレーニング - テスト - 特徴量エンジニアリングのサイクルでは、モデルに対してホールドアウトまたはテストセットを検査します。その場合、それぞれの結果を利用してメトリクスを計算できます。先ほどのようにエンドポイントをデプロイすることもできますが、その場合は必ずエンドポイントを削除してください。ただし、もっと効率の良い方法があります。

モデルの変換メソッドを使用して、transformer オブジェクトを取得します。そうすると、このオブジェクトの変換メソッドを使用して、テスト用データセット全体で予測を実行できます。SageMaker は、以下の機能を備えています。 

- モデルを使用してインスタンスをスピンアップする
- すべての入力値に対して予測を実行する
- それらの値を Amazon Simple Storage Service (Amazon S3) に書き込む 
- インスタンスを終了する

まず、transformer オブジェクトが入力値として取り込めるよう、データを CSV ファイルに変換します。今回は、**iloc** を使用してすべての行を取得し、先頭列を *除く* すべての列を取得します。


In [12]:
# テストデータから特徴量を抽出し、1列目以降を取得
batch_X = 

# 取得したデータの最初の5行を表示


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
415,11.89,21.17,76.39,433.8,0.09773,0.08120,0.02555,0.02179,0.2019,0.06290,...,13.050,27.21,85.09,522.9,0.14260,0.21870,0.11640,0.08263,0.3075,0.07351
235,14.03,21.25,89.79,603.4,0.09070,0.06945,0.01462,0.01896,0.1517,0.05835,...,15.330,30.28,98.27,715.5,0.12870,0.15130,0.06231,0.07963,0.2226,0.07617
568,7.76,24.54,47.92,181.0,0.05263,0.04362,0.00000,0.00000,0.1587,0.05884,...,9.456,30.37,59.16,268.6,0.08996,0.06444,0.00000,0.00000,0.2871,0.07039
488,11.68,16.17,75.49,420.5,0.11280,0.09263,0.04279,0.03132,0.1853,0.06401,...,13.320,21.59,86.57,549.8,0.15260,0.14770,0.14900,0.09815,0.2804,0.08024
14,13.73,22.61,93.60,578.3,0.11310,0.22930,0.21280,0.08025,0.2069,0.07682,...,15.030,32.01,108.80,697.7,0.16510,0.77250,0.69430,0.22080,0.3596,0.14310


次に、データを CSV ファイルに書き込みます。

In [13]:
# バッチデータの入力ファイル名を指定
batch_X_file=

# バッチデータをS3にアップロード


最後は、変換を実行する前に、入力ファイル、出力場所、インスタンスタイプを指定して transformer を設定します。
**注:** この処理は最長 5 分かかることがあります。

In [14]:
# バッチ変換の出力先を指定


# バッチ変換の入力データのパスを指定


# XGBoostトランスフォーマーを作成
# instance_count: トランスフォーマーの実行に使用するインスタンスの数
# instance_type: トランスフォーマーのインスタンスのタイプ
# strategy: バッチ変換のストラテジーを指定（MultiRecordは複数の入力レコードに対応）
# assemble_with: バッチ変換の際に結果をどのようにまとめるか指定（Lineは行ごとにまとめる）
# output_path: バッチ変換の結果を保存するS3のパスを指定
xgb_transformer = xgb_model.transformer()

# バッチ変換を実行
# data: 変換対象のデータのパスを指定
# data_type: 変換対象のデータのタイプを指定（S3PrefixはS3上のプレフィックス内のデータを指定）
# content_type: 変換対象データのコンテンツタイプを指定
# split_type: 入力データをどのように分割するか指定（Lineは行ごとに分割）
xgb_transformer.transform()

# バッチ変換が完了するのを待機


# バッチ変換完了が完了したことを表示
print('Batch transformation completed!')

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2024-01-07-01-55-05-895
INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2024-01-07-01-55-06-593


.................................
.[2024-01-07:02:00:40:INFO] No GPUs detected (normal if no gpus installed)
[2024-01-07:02:00:40:INFO] No GPUs detected (normal if no gpus installed)
[2024-01-07:02:00:40:INFO] nginx config: 
worker_processes auto;
[2024-01-07:02:00:40:INFO] No GPUs detected (normal if no gpus installed)
[2024-01-07:02:00:40:INFO] No GPUs detected (normal if no gpus installed)
[2024-01-07:02:00:40:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) {
      proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
      proxy_set_header H

変換が完了したら、Amazon S3 から結果をダウンロードし、入力と比較します。

まず、Amazon S3 から出力をダウンロードし、pandas の DataFrame にロードします。


In [15]:
# boto3モジュールを使用してS3クライアントを作成
s3 = 

# S3バケットからオブジェクトを取得
obj = 

# バッチ変換の結果として得られた予測結果を読み込む
# ここではCSVデータを読み込み、列名を指定してDataFrameに変換
target_predicted = 

# DataFrameの最初の5行を表示


,target
0,0.997857
1,0.994858
2,0.994762
3,0.999401
4,0.011977


関数を使用すると、確率を *0* または *1* のいずれかに変換できます。

最初のテーブル出力は *予測値*、2 番目のテーブル出力は *元のテストデータ* になります。

In [16]:
# 閾値を設定して二値変換する関数を定義
# binary_convert関数では、確率が指定された閾値（ここでは0.65）より大きい場合は1、それ以外は0に変換されます。
def binary_convert(x):
    threshold = 0.65
    if x > threshold:
        return 1
    else:
        return 0

# 'target'列の各要素に二値変換を適用し、新しい 'binary' 列を作成


# 最初の10行を表示して、変換が正しく適用されていることを確認


# テストデータの最初の10行を表示


     target  binary
0  0.997857       1
1  0.994858       1
2  0.994762       1
3  0.999401       1
4  0.011977       0
5  0.809007       1
6  0.698530       1
7  0.998061       1
8  0.001618       0
9  0.999521       1


,target,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
415,1,11.89,21.17,76.39,433.8,0.09773,0.08120,0.02555,0.02179,0.2019,...,13.050,27.21,85.09,522.9,0.14260,0.21870,0.11640,0.08263,0.3075,0.07351
235,1,14.03,21.25,89.79,603.4,0.09070,0.06945,0.01462,0.01896,0.1517,...,15.330,30.28,98.27,715.5,0.12870,0.15130,0.06231,0.07963,0.2226,0.07617
568,1,7.76,24.54,47.92,181.0,0.05263,0.04362,0.00000,0.00000,0.1587,...,9.456,30.37,59.16,268.6,0.08996,0.06444,0.00000,0.00000,0.2871,0.07039
488,1,11.68,16.17,75.49,420.5,0.11280,0.09263,0.04279,0.03132,0.1853,...,13.320,21.59,86.57,549.8,0.15260,0.14770,0.14900,0.09815,0.2804,0.08024
14,0,13.73,22.61,93.60,578.3,0.11310,0.22930,0.21280,0.08025,0.2069,...,15.030,32.01,108.80,697.7,0.16510,0.77250,0.69430,0.22080,0.3596,0.14310
375,1,16.17,16.07,106.30,788.5,0.09880,0.14380,0.06651,0.05397,0.1990,...,16.970,19.14,113.10,861.5,0.12350,0.25500,0.21140,0.12510,0.3153,0.08960
41,0,10.95,21.35,71.90,371.1,0.12270,0.12180,0.10440,0.05669,0.1895,...,12.840,35.34,87.22,514.0,0.19090,0.26980,0.40230,0.14240,0.2964,0.09606
288,1,11.26,19.96,73.72,394.1,0.08020,0.11810,0.09274,0.05588,0.2595,...,11.860,22.33,78.27,437.6,0.10280,0.18430,0.15460,0.09314,0.2955,0.07009
198,0,19.18,22.49,127.50,1148.0,0.08523,0.14280,0.11140,0.06772,0.1767,...,23.360,32.06,166.40,1688.0,0.13220,0.56010,0.38650,0.17080,0.3193,0.09221
296,1,10.91,12.35,69.14,363.7,0.08518,0.04721,0.01236,0.01369,0.1449,...,11.370,14.82,72.42,392.2,0.09312,0.07506,0.02884,0.03194,0.2143,0.06643


**注:** **binary_convert** 関数の *しきい値* は *.65* に設定されています。

**チャレンジ課題:** しきい値の値を変えて実験してみましょう。結果に影響するでしょうか?

**注:** 最初のモデルは精度が良くないこともあります。次回のラボでいくつかメトリックを生成してから、最後のラボでモデルを調整します。

# お疲れ様でした。

このラボを完了しました。ラボガイドの手順に従ってラボを終了してください。